## AutoCast processor training example

This notebook demonstrates training a processor directly on encoded data.

### Example dataaset

We use the `ReactionDiffusion` dataset as an example dataset to illustrate training and evaluation of models. This dataset simulates the advection-diffusion equation in 2D.


In [ ]:
from autocast.data.encoded_dataset import MiniWellDataModule, MiniWellInputOutput
from autocast.metrics.spatiotemporal import MAE, MSE, RMSE, VRMSE
from einops import rearrange, repeat
import torch
from pathlib import Path
import h5py
from torch.utils.data import ConcatDataset, Dataset, DataLoader
from autocast.types.batch import collate_encoded_samples

THE_WELL = False
simulation_name = "rayleigh_benard"
n_steps_input = 1
n_steps_output = 4
stride = 1
rollout_stride = 4

class PreprocessedInMemoryDataset(Dataset):
    def __init__(self, file_path, steps, stride, label_mean=None, label_std=None):
        with h5py.File(file_path, "r") as f:
            # Load raw data
            # shape: (Traj, T, C, H, W) e.g. (40, 200, 64, 16, 4)
            # Rearrange immediately to (Traj, T, W, H, C) for faster slicing later
            state_raw = torch.from_numpy(f["state"][:])
            self.state = rearrange(state_raw, "b t c h w -> b t w h c")
            
            # Load and normalize labels
            label_raw = torch.from_numpy(f["label"][:])
            if label_mean is not None and label_std is not None:
                label_raw = (label_raw - label_mean) / label_std
            self.label = label_raw
        
        self.trajectories = self.state.shape[0]
        self.steps_per_trajectory = self.state.shape[1]
        self.steps = steps
        self.stride = stride
        
        # Pre-calculate spatial expansion shape for labels to save time in getitem
        # self.state is (B, T, W, H, C)
        self.spatial_dims = self.state.shape[2:4] # (W, H)
        
    def __len__(self):
        return self.trajectories * (
            self.steps_per_trajectory - (self.steps - 1) * self.stride
        )

    def __getitem__(self, i):
        crops_per_trajectory = (
            self.steps_per_trajectory - (self.steps - 1) * self.stride
        )
        i, j = i // crops_per_trajectory, i % crops_per_trajectory
        
        # Slice state: (T_slice, W, H, C)
        # Slicing on dim 1 (time) which is contiguous-ish after dim 0 (traj)
        state_slice = self.state[
            i, j : j + (self.steps - 1) * self.stride + 1 : self.stride
        ]
        
        return {
            "state": state_slice,
            "label": self.label[i],
            "encoded_info": {} 
        }

class NormalizedMiniWellDataset(MiniWellInputOutput):
    def __init__(self, filepath, n_steps_input, n_steps_output, stride=1, concat_inputs_and_label=True, label_mean=None, label_std=None):
        Dataset.__init__(self)
        
        self.n_steps_input = n_steps_input
        self.n_steps_output = n_steps_output
        self.concat_inputs_and_label = concat_inputs_and_label
        
        # We pass norm stats to the sub-dataset to handle it once at load time
        l_mean = torch.tensor(label_mean) if label_mean is not None else None
        l_std = torch.tensor(label_std) if label_std is not None else None
        
        steps = n_steps_input + n_steps_output
        
        if isinstance(filepath, str):
            filepath = [filepath]
            
        print("Loading and preprocessing datasets into memory...")
        datasets = [PreprocessedInMemoryDataset(f, steps, stride, l_mean, l_std) for f in filepath]
        self.miniwell_dataset = ConcatDataset(datasets)
        print("Done loading.")

    def __len__(self):
        return len(self.miniwell_dataset)

    def __getitem__(self, index):
        data = self.miniwell_dataset.__getitem__(index)
        
        # Data is already in (T, W, H, C) format
        full_state = data["state"]
        
        input_fields = full_state[: self.n_steps_input]
        output_fields = full_state[self.n_steps_input : self.n_steps_input + self.n_steps_output]
        label = data["label"]

        if self.concat_inputs_and_label:
            # Expand label and concat
            # label: (C_label)
            # input_fields: (T_in, W, H, C_in)
            t = input_fields.shape[0]
            w, h = input_fields.shape[1], input_fields.shape[2]
            
            # Efficient implementation
            # (C) -> (1, 1, 1, C) -> expand -> (T, W, H, C)
            label_expanded = label.view(1, 1, 1, -1).expand(t, w, h, -1)
            
            # This cat is the only heavy op remaining per-sample
            input_fields = torch.cat([input_fields, label_expanded], dim=-1)

        return self.to_sample(
            {
                "input_fields": input_fields,
                "output_fields": output_fields,
                "label": label,
                "encoded_info": data.get("encoded_info", {}),
            }
        )

class CustomMiniWellDataModule(MiniWellDataModule):
    def __init__(
        self,
        dataset_cls=None,
        dataset_kwargs=None,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.dataset_cls = dataset_cls or MiniWellInputOutput
        self.dataset_kwargs = dataset_kwargs or {}

    def _create_dataset(self, dir_path):
        if not dir_path.exists():
            return None

        files = sorted(dir_path.glob("*.h5")) + sorted(dir_path.glob("*.hdf5"))
        if not files:
            return None

        common_kwargs = {
            "n_steps_input": self.n_steps_input,
            "n_steps_output": self.n_steps_output,
            "stride": self.stride,
            "concat_inputs_and_label": self.concat_inputs_and_label,
        }
        common_kwargs.update(self.dataset_kwargs)

        if len(files) == 1:
            return self.dataset_cls(filepath=str(files[0]), **common_kwargs)

        return self.dataset_cls(filepath=[str(f) for f in files], **common_kwargs)
    
    def train_dataloader(self):
        if self.train_dataset is None:
            self.setup(stage="fit")
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers, 
            collate_fn=collate_encoded_samples,
            pin_memory=True, # Faster transfer to GPU
            persistent_workers=True, # Keep workers alive
        )

base_path = (
    f"../datasets/{simulation_name}/1e3z5x2c_{simulation_name}_dcae_f32c64_large/cache/{simulation_name}"
)

# Stats for Rayleigh Benard labels
label_mean = 3.6702
label_std = 6.6332

datamodule = CustomMiniWellDataModule(
    data_path=base_path,
    n_steps_input=n_steps_input, n_steps_output=n_steps_output, stride=stride,
    batch_size=64, # Increase batch size if GPU util is still low (e.g. 128 or 256)
    num_workers=4,
    dataset_cls=NormalizedMiniWellDataset,
    dataset_kwargs={"label_mean": label_mean, "label_std": label_std}
)

### Set-up logging


In [ ]:
from autocast.logging import maybe_watch_model
from autocast.logging.wandb import create_notebook_logger

logger, watch = create_notebook_logger(
    project="autocast-notebooks",
    name=f"06_processor_{simulation_name}",
    tags=["notebook", simulation_name],
    enabled=True,
)

In [ ]:
batch = next(iter(datamodule.train_dataloader()))
n_channels = batch.encoded_inputs.shape[-1]
w, h = batch.encoded_inputs.shape[2:4]


### Example shape and batch


In [ ]:
datamodule.train_dataset[0].encoded_inputs.shape

In [ ]:
batch = next(iter(datamodule.train_dataloader()))

batch.encoded_inputs.shape

In [ ]:

from azula.noise import VPSchedule

from autocast.models.processor import ProcessorModel
from autocast.nn.unet import TemporalUNetBackbone
from autocast.nn.vit import TemporalViTBackbone
from autocast.processors.flow_matching import FlowMatchingProcessor

batch = next(iter(datamodule.train_dataloader()))
n_channels = batch.encoded_inputs.shape[-1]
print(f"Batch Stats - Input Mean: {batch.encoded_inputs.mean():.3f}, Std: {batch.encoded_inputs.std():.3f}")
print(f"Batch Stats - Output Mean: {batch.encoded_output_fields.mean():.3f}, Std: {batch.encoded_output_fields.std():.3f}")

processor_name = "flow_matching"  # set to "diffusion" to compare
# processor_name = "diffusion"  # set to "flow_matching" to compare
n_latent_in = batch.encoded_inputs.shape[-1]
n_latent_out = batch.encoded_output_fields.shape[-1]

# Reduced model size for stability/performance on small data
backbone = TemporalViTBackbone(
    in_channels=n_latent_out,
    out_channels=n_latent_out,
    cond_channels=n_latent_in,
    n_steps_output=n_steps_output,
    n_steps_input=n_steps_input,
    mod_features=256,
    hid_channels=768, # Increased capacity
    hid_blocks=12,    # Increased capacity
    temporal_method="attention", # Better for sequence handling
    attention_heads=12,
    spatial=2,
    patch_size=1,     # CRITICAL: spatial dim is 4x16, default=4 would crush it to 1x4 tokens
    dropout=0.05,
    ffn_factor=4,
    checkpointing=True,
)

if processor_name == "flow_matching":
    processor = FlowMatchingProcessor(
        backbone=backbone,
        schedule=VPSchedule(),
        n_steps_output=n_steps_output,
        n_channels_out=n_latent_out,
        stride=stride,
        flow_ode_steps=25, 
    )
else:
    from autocast.processors.diffusion import DiffusionProcessor

    processor = DiffusionProcessor(
        backbone=backbone,
        schedule=VPSchedule(),
        n_steps_output=n_steps_output,
        n_channels_out=n_latent_out,
    )

model = ProcessorModel(
    processor=processor,
    learning_rate=3e-4, # Slightly higher LR
    val_metrics=[VRMSE(), MSE(), MAE(), RMSE()],
    test_metrics=[VRMSE(), MSE(), MAE(), RMSE()],
)
maybe_watch_model(logger, model, watch)


In [ ]:
model.val_metrics

In [ ]:
model(batch.encoded_inputs).shape

### Run trainer


In [ ]:
import lightning as L

# device = "mps"  # "cpu"
device = "cuda"  # "cpu"
# Removed limit_train_batches=0.1 to use full data
trainer = L.Trainer(
    max_epochs=10, accelerator=device, logger=logger,
    limit_train_batches=1.0, limit_val_batches=0.1
)
trainer.fit(model, datamodule)
trainer.save_checkpoint(f"./{simulation_name}_{processor_name}_model.ckpt")


### Run the evaluation


In [ ]:
trainer.test(model, datamodule)